In [1]:
# from google.colab import drive; drive.mount('/content/drive')
base_path = './' ## change to 'drive/MyDrive/POC2PROD/' for Colab

In [ ]:
import pandas as pd
import torch
import re
import os
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt

In [ ]:
# Load data
df = pd.read_csv(os.path.join(base_path, 'stackoverflow_posts.csv'))
if 'tag_id' not in df.columns:
    tag_map = {t: i for i, t in enumerate(sorted(df['tag_name'].dropna().unique()))}
    df['tag_id'] = df['tag_name'].map(tag_map)
df = df.dropna(subset=['title', 'tag_id'])
df.head()

,post_id,tag_name,tag_id,tag_position,title
0,1987528,php,5,0,Is it possible to execute the procedure of a f...
1,1987531,ruby-on-rails,4984,0,ruby on rails: how to change BG color of optio...
2,1987531,list,5608,1,ruby on rails: how to change BG color of optio...
3,1987531,select,1151,2,ruby on rails: how to change BG color of optio...
4,1987538,c#,9,0,How to read tags out of m4a files in .NET?


In [ ]:
# Preprocess text
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    return text
df['clean_title'] = df['title'].astype(str).apply(preprocess_text)
df.head()

,post_id,tag_name,tag_id,tag_position,title,clean_title
0,1987528,php,5,0,Is it possible to execute the procedure of a f...,is it possible to execute the procedure of a f...
1,1987531,ruby-on-rails,4984,0,ruby on rails: how to change BG color of optio...,ruby on rails how to change bg color of option...
2,1987531,list,5608,1,ruby on rails: how to change BG color of optio...,ruby on rails how to change bg color of option...
3,1987531,select,1151,2,ruby on rails: how to change BG color of optio...,ruby on rails how to change bg color of option...
4,1987538,c#,9,0,How to read tags out of m4a files in .NET?,how to read tags out of ma files in net


In [ ]:
# Remove rare labels
label_counts = df['tag_id'].value_counts()
rare_labels = label_counts[label_counts < 3].index
if len(rare_labels) > 0:
    df = df[~df['tag_id'].isin(rare_labels)]

# Split data for stage 1: only tag_position = 0
df_stage1 = df[df['tag_position'] == 0].copy()
train_df_stage1, test_df_stage1 = train_test_split(df_stage1, test_size=0.2, stratify=df_stage1['tag_id'], random_state=42)

# Split data for stage 2: full dataset
train_df_full, test_df_full = train_test_split(df, test_size=0.2, stratify=df['tag_id'], random_state=42)

In [ ]:
# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
label_dict = {label: i for i, label in enumerate(train_df_stage1['tag_id'].unique())}
num_labels = len(label_dict)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_labels)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Create dataset class
class Dataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, label_dict):
        self.df = df
        self.tokenizer = tokenizer
        self.label_dict = label_dict
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        text = self.df.iloc[idx]['clean_title']
        label = self.label_dict[self.df.iloc[idx]['tag_id']]
        encoding = self.tokenizer(text, truncation=True, padding='max_length', max_length=128, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset_stage1 = Dataset(train_df_stage1, tokenizer, label_dict)
test_dataset_stage1 = Dataset(test_df_stage1, tokenizer, label_dict)

In [ ]:
# Stage 1: Train on tag_position = 0 subset
training_args = TrainingArguments(
    output_dir=os.path.join(base_path, 'results_stage1'),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir=os.path.join(base_path, 'logs_stage1'),
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_stage1,
    eval_dataset=test_dataset_stage1,
)
trainer.train()

C:\Users\mrnew\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
# Evaluate stage 1
predictions = trainer.predict(test_dataset_stage1)
pred_labels = [list(label_dict.keys())[p] for p in predictions.predictions.argmax(axis=1)]
true_labels = test_df_stage1['tag_id'].tolist()
acc = accuracy_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels, average='weighted')
print(f'Stage 1 - Test acc: {acc:.4f}, f1: {f1:.4f}')
print(classification_report(true_labels, pred_labels))

In [ ]:
# Update label dict for full dataset
label_dict_full = {label: i for i, label in enumerate(train_df_full['tag_id'].unique())}
num_labels_full = len(label_dict_full)

# Load new model for stage 2
model_stage2 = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_labels_full)

train_dataset_full = Dataset(train_df_full, tokenizer, label_dict_full)
test_dataset_full = Dataset(test_df_full, tokenizer, label_dict_full)

In [ ]:
# Stage 2: Train on full dataset
training_args_full = TrainingArguments(
    output_dir=os.path.join(base_path, 'results_stage2'),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir=os.path.join(base_path, 'logs_stage2'),
)
trainer_full = Trainer(
    model=model_stage2,
    args=training_args_full,
    train_dataset=train_dataset_full,
    eval_dataset=test_dataset_full,
)
trainer_full.train()

In [ ]:
# Evaluate stage 2
predictions_full = trainer_full.predict(test_dataset_full)
pred_labels_full = [list(label_dict_full.keys())[p] for p in predictions_full.predictions.argmax(axis=1)]
true_labels_full = test_df_full['tag_id'].tolist()
acc_full = accuracy_score(true_labels_full, pred_labels_full)
f1_full = f1_score(true_labels_full, pred_labels_full, average='weighted')
print(f'Stage 2 - Test acc: {acc_full:.4f}, f1: {f1_full:.4f}')
print(classification_report(true_labels_full, pred_labels_full))